In [1]:
!pip -q install kaggle pyarrow scikit-learn tqdm


In [2]:
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle ready")


Kaggle ready


In [3]:
!kaggle competitions download -c sem-eval-2026-task-13-subtask-a
!unzip -q sem-eval-2026-task-13-subtask-a.zip
print(" Data ready")

 93% 419M/448M [00:06<00:00, 67.2MB/s]
100% 448M/448M [00:06<00:00, 67.9MB/s]
 Data ready


In [4]:
import pandas as pd

train_df = pd.read_parquet("Task_A/train.parquet")
test_df = pd.read_parquet("Task_A/test.parquet")
test_sample_df = pd.read_parquet("Task_A/test_sample.parquet")

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Test_sample:", test_sample_df.shape)


Train: (500000, 4)
Test: (500000, 2)
Test_sample: (1000, 4)


In [5]:
train_sub = train_df.sample(n=80000, random_state=42).reset_index(drop=True)

X_train = train_sub["code"].astype(str).reset_index(drop=True)
y_train = train_sub["label"].values

X_ts = test_sample_df["code"].astype(str).reset_index(drop=True)
y_ts = test_sample_df["label"].values

print("Train:", len(X_train), "Test_sample:", len(X_ts))


Train: 80000 Test_sample: 1000


In [6]:
import numpy as np

def style_features(code: str):
    if not isinstance(code, str):
        code = ""
    n = len(code)
    lines = code.count("\n") + 1
    avg_line = n / lines if lines > 0 else 0

    ws = sum(c.isspace() for c in code)
    digits = sum(c.isdigit() for c in code)
    punct = sum((not c.isalnum()) and (not c.isspace()) for c in code)

    ws_r = ws / (n + 1e-6)
    dig_r = digits / (n + 1e-6)
    punc_r = punct / (n + 1e-6)

    comment_cnt = code.count("#") + code.count("//") + code.count("/*")

    return np.array([n, lines, avg_line, comment_cnt, punc_r, dig_r], dtype=np.float32)

def build_style_matrix(series):
    return np.vstack([style_features(x) for x in series])

X_train_style = build_style_matrix(X_train)
X_ts_style = build_style_matrix(X_ts)
print("✅ Style matrix shapes:", X_train_style.shape, X_ts_style.shape)

✅ Style matrix shapes: (80000, 6) (1000, 6)


In [14]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans

K = 4

style_scaler = StandardScaler()
X_train_style_s = style_scaler.fit_transform(X_train_style)
X_ts_style_s = style_scaler.transform(X_ts_style)

kmeans = MiniBatchKMeans(n_clusters=K, random_state=42, batch_size=4096)
train_cluster = kmeans.fit_predict(X_train_style_s)
ts_cluster = kmeans.predict(X_ts_style_s)

print("✅ Cluster distribution (train):", np.bincount(train_cluster))
print("✅ Cluster distribution (test_sample):", np.bincount(ts_cluster))


✅ Cluster distribution (train): [26154  6261 15100 32485]
✅ Cluster distribution (test_sample): [737  12  38 213]


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    min_df=5,
    max_features=120_000
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_ts_tfidf = tfidf.transform(X_ts)

print("TF-IDF shapes:", X_train_tfidf.shape, X_ts_tfidf.shape)


TF-IDF shapes: (80000, 120000) (1000, 120000)


In [16]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack, csr_matrix

experts = []

X_train_num_sp = csr_matrix(X_train_style_s)
X_ts_num_sp = csr_matrix(X_ts_style_s)

for c in range(K):
    idx = np.where(train_cluster == c)[0]
    print(f"\n--- Training expert {c} | samples: {len(idx)} ---")

    Xc_tfidf = X_train_tfidf[idx]
    yc = y_train[idx]

    nb = MultinomialNB(alpha=0.5)
    nb.fit(Xc_tfidf, yc)

    Xc_comb = hstack([Xc_tfidf, X_train_num_sp[idx]])
    lr = LogisticRegression(max_iter=3000, n_jobs=-1)
    lr.fit(Xc_comb, yc)

    experts.append((nb, lr))

print("\n All experts trained")



--- Training expert 0 | samples: 26154 ---

--- Training expert 1 | samples: 6261 ---

--- Training expert 2 | samples: 15100 ---

--- Training expert 3 | samples: 32485 ---

 All experts trained


In [18]:
def length_bucket(code: str):
    n = len(code)
    if n <= 250:
        return 0  # short
    elif n <= 1200:
        return 1  # medium
    else:
        return 2  # long

ts_len_bucket = np.array([length_bucket(x) for x in X_ts], dtype=int)
print("Length bucket distribution (test_sample):", np.bincount(ts_len_bucket))


Length bucket distribution (test_sample): [ 95 546 359]


In [19]:
import numpy as np
from sklearn.metrics import f1_score
from scipy.sparse import hstack

thresholds = np.linspace(0.70, 0.99, 50)

T = {}

for c in range(K):
    for b in range(3):
        idx = np.where((ts_cluster == c) & (ts_len_bucket == b))[0]
        if len(idx) < 15:
            continue

        nb, lr = experts[c]

        p_nb = nb.predict_proba(X_ts_tfidf[idx])[:, 1]
        Xc_comb_ts = hstack([X_ts_tfidf[idx], X_ts_num_sp[idx]])
        p_lr = lr.predict_proba(Xc_comb_ts)[:, 1]
        p_avg = (p_nb + p_lr) / 2

        best_t, best_f = 0.95, -1
        for t in thresholds:
            preds = (p_avg >= t).astype(int)
            f = f1_score(y_ts[idx], preds, average="macro")
            if f > best_f:
                best_f = f
                best_t = float(t)

        T[(c, b)] = best_t
        print(f"Cluster {c} | bucket {b} | n={len(idx)} | best_t={best_t:.3f} | f1={best_f:.4f}")


fallback_cluster = {}
for c in range(K):
    vals = [T[(c,b)] for b in range(3) if (c,b) in T]
    if len(vals) > 0:
        fallback_cluster[c] = float(np.median(vals))

fallback_global = float(np.median(list(T.values()))) if len(T) > 0 else 0.95

print("\n Threshold table size:", len(T))
print(" Fallback per cluster:", fallback_cluster)
print(" Global fallback:", fallback_global)


Cluster 0 | bucket 0 | n=35 | best_t=0.783 | f1=0.6830
Cluster 0 | bucket 1 | n=356 | best_t=0.972 | f1=0.6854
Cluster 0 | bucket 2 | n=346 | best_t=0.984 | f1=0.5876
Cluster 2 | bucket 1 | n=22 | best_t=0.836 | f1=0.5810
Cluster 3 | bucket 0 | n=44 | best_t=0.949 | f1=0.7708
Cluster 3 | bucket 1 | n=162 | best_t=0.966 | f1=0.6000

 Threshold table size: 6
 Fallback per cluster: {0: 0.9722448979591837, 2: 0.8361224489795918, 3: 0.9574489795918367}
 Global fallback: 0.9574489795918367


In [20]:
#dont run for ablation 1 or ablation 2 orr 3
import numpy as np
from scipy.sparse import hstack
from sklearn.metrics import f1_score

preds_all = np.zeros(len(X_ts), dtype=int)

for c in range(K):
    idx_c = np.where(ts_cluster == c)[0]
    if len(idx_c) == 0:
        continue

    nb, lr = experts[c]


    p_nb = nb.predict_proba(X_ts_tfidf[idx_c])[:, 1]
    Xc_comb_ts = hstack([X_ts_tfidf[idx_c], X_ts_num_sp[idx_c]])
    p_lr = lr.predict_proba(Xc_comb_ts)[:, 1]
    p_avg = (p_nb + p_lr) / 2


    for j, global_idx in enumerate(idx_c):
        b = ts_len_bucket[global_idx]
        t = T.get((c, b), fallback_cluster.get(c, fallback_global))
        preds_all[global_idx] = int(p_avg[j] >= t)

print(" MoE + length-bucket MacroF1 on test_sample:", f1_score(y_ts, preds_all, average="macro"))


 MoE + length-bucket MacroF1 on test_sample: 0.6399626517273576


In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.sparse import hstack, csr_matrix

BATCH = 20000

X_test = test_df["code"].astype(str).reset_index(drop=True)
n = len(X_test)

final_labels = []

for start in tqdm(range(0, n, BATCH)):
    end = min(start + BATCH, n)
    batch = X_test.iloc[start:end]


    batch_style = build_style_matrix(batch)
    batch_style_s = style_scaler.transform(batch_style)
    batch_cluster = kmeans.predict(batch_style_s)

    batch_bucket = np.array([length_bucket(x) for x in batch.tolist()], dtype=int)


    batch_tfidf = tfidf.transform(batch)
    batch_num_sp = csr_matrix(batch_style_s)

    batch_preds = np.zeros(len(batch), dtype=int)

    for c in range(K):
        idx = np.where(batch_cluster == c)[0]
        if len(idx) == 0:
            continue

        nb, lr = experts[c]
        p_nb = nb.predict_proba(batch_tfidf[idx])[:, 1]
        Xc_comb = hstack([batch_tfidf[idx], batch_num_sp[idx]])
        p_lr = lr.predict_proba(Xc_comb)[:, 1]
        p_avg = (p_nb + p_lr) / 2


        for j, local_idx in enumerate(idx):
            b = batch_bucket[local_idx]
            t = T.get((c, b), fallback_cluster.get(c, fallback_global))
            batch_preds[local_idx] = int(p_avg[j] >= t)

    final_labels.append(batch_preds)

final_labels = np.concatenate(final_labels)

print(" Done predictions:", final_labels.shape)
print("Label distribution:", np.bincount(final_labels))


  0%|          | 0/25 [00:00<?, ?it/s]

In [ ]:
submission = pd.DataFrame({
    "ID": test_df["ID"],
    "label": final_labels
})

submission.to_csv("submission_moe_k4_lenbucket.csv", index=False)
print("✅ Saved submission_moe_k4_lenbucket.csv")
submission.head()
